In [2]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-6-luna",
        input=prompt
    )
    return response.output_text

In [4]:
llm("Hey, what's up?")

"Hey! Not much—I'm here and ready to help. What’s up with you?"

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Possibly! It depends on the course’s enrollment dates and whether late registration is allowed. Check the course page for an **Enroll/Join** option, or tell me which course you mean and I can help you figure it out.


In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
answer = llm(prompt)
print(answer)

Yes, you can join now. If you want a certificate, submit your project while submissions are still open.


In [9]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [10]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1456

In [11]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [12]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [13]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [14]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Does the course certificate show the number of course hours?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?']

In [15]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

In [16]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [17]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Course: How do I start?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?']

In [18]:
question

'I just discovered the course. Can I join now?'

In [19]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [20]:
search_results = search(question)

In [21]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [22]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [23]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [24]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [25]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [26]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI w

In [27]:
response = openai_client.responses.create(
    model="gpt-6-luna",
    input=prompt
)

In [31]:
response

Response(id='resp_046aabe5e7edfb07006ae0dd39891c87d2a7a39e7d55636b0f', access_programs=None, created_at=1793121593.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-6-luna', object='response', output=[ResponseReasoningItem(id='rs_046aabe5e7edfb07006ae0dd3a6e6087d2962ab395b2b495da', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABq4N07jAZravpO6jIifFOePs8pQwt5PzE11ctE8gkZJ35A2OQ-oazQTJ1Z7wn3c1H7pO6b3VwiC2QmLcDryGYoubDAkTDyOdoqhTMHfnMUCqTAVQ2M9QDmB6V9jaZFN6rXNMe5QFJ51fgki5ZimJOzjiIomuUeu9qbR_HrT7nIhT4AxfrQSXxo5WNmO0kP9R88UhIWzgjaq8hJpVGGBR_xjtXT3v44RuwegvAXLaSRD28iA6JLUgdrMTHtgvlzVSogph09sncGXzULYBfx_jvuhjg0NUAZ9hD7flXDH-wLBDibpypuvqJrf1h7ImlkzdmfvoGpTwWJoSC_PYlQPi3W7Q8TRCwXacv0FVVFQPD3gi70zfvxC0HAPIyXGb2csSl5t0Oit2Cao7b8LVW6emhiceYZz-HJDo1szLBqnOHTL6jvYcMC4SB-En3rfzVkH6DARP1aO3w9tS8szMFDEUvexJosSG1qpg-AD85r0Oyqf_kFBYqZUcOTk_2U0JzsvDURJjIqLPOrSkpZZ1sfhrAyJXlq5v5Cal5kE54QCvMyWeNwjg5Op5j4vIEPlyB6Ue3w8ZMmGpFb0UH45cwjEaAjDoGZKDJEN9fRl7Epzoh

In [32]:
response.output[1]

ResponseOutputMessage(id='msg_046aabe5e7edfb07006ae0dd3b1a7087d2a41a6f117d739fd0', content=[ResponseOutputText(annotations=[], text='Yes—you can join and start learning now; registration isn’t required. To earn a certificate, you’ll need to submit your capstone project and complete the required peer reviews while a live cohort is accepting submissions.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [33]:
response.output[1].content[0].text

'Yes—you can join and start learning now; registration isn’t required. To earn a certificate, you’ll need to submit your capstone project and complete the required peer reviews while a live cohort is accepting submissions.'

In [34]:
response.output_text

'Yes—you can join and start learning now; registration isn’t required. To earn a certificate, you’ll need to submit your capstone project and complete the required peer reviews while a live cohort is accepting submissions.'

In [35]:
response.usage

ResponseUsage(input_tokens=506, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=109, output_tokens_details=OutputTokensDetails(reasoning_tokens=61), total_tokens=615)

In [36]:
input_price = 0.1 / 1_000_000
output_price = 0.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00010510000000000001

In [37]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-6-luna",
    input=message_history
)

In [38]:
def llm(instructions, user_prompt, model="gpt-6-luna"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [39]:
prompt

'Question:\nI just discovered the course. Can I join now?\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nGeneral Course-Related Questions\nQ: Does the course certificate show the number of course hours?\nA: No. The certificate does not state a total number of hours.\n\nModule 3: Orchestration\nQ: Why do we need orchestration / Kestra — can\'t I just run the code in a notebook?\nA: Notebooks are great for learning and experi

In [40]:
def rag(query, model="gpt-6-luna"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [41]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes. You can join now. If you want a certificate, you’ll need to submit your capstone project and complete the required peer reviews while a live cohort is accepting submissions. Homework isn’t required for the certificate.


In [42]:
rag("How do I get a certificate?")

'To get a certificate, you need to complete the course with a **live cohort**, pass the capstone project, and finish all peer reviews assigned to you by their deadline. Homework is optional.\n\nCertificates are generated manually in one batch after the cohort’s final project attempt and peer-review period are over—you don’t need to request one. Make sure your profile has the name you want printed. You’ll receive an email when the certificate is ready; check your spam folder if you don’t see it.'

In [1]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join and start learning now. If you want a certificate, you’ll need to submit your capstone project and complete the required peer reviews while a live cohort is accepting submissions.


In [2]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [6]:
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join after the course has started. To receive a certificate, you’ll need to submit your project while project submissions are still being accepted.'

In [5]:
assistant.rag("I just discovered the course. Can I join now?")

'Yes, you can still join. If you want a certificate, you’ll need to submit your project and complete the required peer reviews while a live cohort is accepting submissions.'